In [1]:
import pathlib

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import equinox as eqx
from jaxtyping import Array, Float, Int, PyTree, PRNGKeyArray

import torch
from torch.utils.data import DataLoader, Subset

import altair as alt

import trapallada.data
import trapallada.model.signal

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# input_directory = pathlib.Path.cwd().parent / "light curves" / "Simulations"
# input_directory = pathlib.Path.cwd().parent / "light curves"
# input_directory = pathlib.Path.cwd()
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [5]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
len(ds)

12983

In [6]:
ds.metadata_df.sample(n=10)

,label,SNR,color,file name
8376,carrier,31.620,None,singlePeriod/carrier_SNR31.62_Simul116.mat
8905,carrier,10.000,None,singlePeriod/carrier_SNR10_Simul929.mat
10086,carrier,1.000,None,singlePeriod/carrier_SNR1_Simul920.mat
8407,carrier,1.000,None,singlePeriod/carrier_SNR1_Simul937.mat
8577,carrier,31.620,None,singlePeriod/carrier_SNR31.62_Simul118.mat
12001,noise,NaN,1.9453,noise/noise_color1.9453.mat
6049,AM,10.000,None,AM/AM_SNR10_Simul674.mat
6125,AM,3.162,None,AM/AM_SNR3.162_Simul212.mat
11957,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul101.mat
2482,FM,31.620,None,FM/FM_SNR31.62_Simul555.mat


In [7]:
i = ds.metadata_df[ds.metadata_df['label'] == 'AM'].index[0]
ds.metadata_df.loc[i]

label                              AM
SNR                              10.0
color                            None
file name    AM/AM_SNR10_Simul927.mat
Name: 4000, dtype: object

In [8]:
y = ds[i][1]
df = pd.DataFrame(y.squeeze(), columns=['signal'])
df


,signal
0,0.833033
1,1.048084
2,0.757383
3,0.813343
4,0.835784
...,...
9995,0.078315
9996,0.066762
9997,0.342989
9998,0.264237


In [9]:
n_rows = 2000
alt.Chart(df.iloc[:n_rows].reset_index()).mark_line().encode(
    x='index', y='signal'
)

alt.Chart(...)

In [10]:
rng_key = jr.PRNGKey(42)

parameters = {
    'training': {
        'splitting': {
            'test_size':  0.2,
            'random_state': 42
        },
        'batch_size': 32,
        'learning_rate': 1e-3,
    },
}

In [11]:
train_idx, validation_idx = train_test_split(
    np.arange(len(ds)), 
    **parameters['training']['splitting'], 
    shuffle=True,
    stratify=ds.labels
)
train_ds, validation_ds = Subset(ds, train_idx), Subset(ds, validation_idx)

# a random key...
key, subkey = jr.split(rng_key)

# ...to compute a seed for a PyTorch `Generator`...
torch_seed = jr.bits(subkey, (1,), dtype=jnp.uint32).item()
torch_rng = torch.Generator().manual_seed(torch_seed)

# ...meant for "batching" in the `DataLoader`'s
train_dl = DataLoader(train_ds, parameters['training']['batch_size'], shuffle=True, generator=torch_rng)
validation_dl = DataLoader(validation_ds, batch_size=parameters['training']['batch_size'], shuffle=False, generator=torch_rng)

The indexes (1st element in the `tuple` for a batch) are used to build a `DataFrame`

In [12]:
batch = next(iter(train_dl))
batch_metadata_df = ds.metadata_df.loc[batch[0]]
df = pd.DataFrame(batch[1].squeeze().T, columns=batch_metadata_df['file name'].str.extract(r'.*/(.*).(?:jpg|mat)')[0])
df

,AM_SNR3.162_Simul373,carrier_SNR3.162_Simul169,FM_SNR10_Simul934,carrier_SNR1_Simul710,FM_SNR10_Simul813,AM_SNR1_Simul157,carrier_SNR10_Simul584,FM_SNR31.62_Simul76,carrier_SNR10_Simul752,AM_SNR10_Simul126,...,carrier_SNR1_Simul194,AM_SNR10_Simul903,carrier_SNR10_Simul424,FM_SNR3.162_Simul448,carrier_SNR31.62_Simul266,AM_SNR1_Simul771,FM_SNR1_Simul212,AM_SNR1_Simul724,AM_SNR31.62_Simul204,AM_SNR10_Simul215
0,0.873530,0.366294,1.213299,-0.780347,1.057264,0.937096,0.188470,0.982803,0.178120,0.980861,...,1.117118,1.190087,0.087181,0.179355,-0.012090,1.552982,1.976153,1.297910,1.009297,0.681374
1,1.379022,-0.554513,1.187360,0.302909,0.968973,0.733333,0.311995,0.987700,0.067379,1.089693,...,0.482503,1.141025,0.119563,0.165130,0.006320,0.736538,1.273790,1.972451,1.002199,1.156233
2,1.082426,0.424128,0.758747,-0.734449,1.125389,1.082168,-0.789615,0.971781,-0.190970,0.905229,...,0.103971,0.856297,0.093810,0.633183,0.012509,0.613180,-0.321107,0.810956,1.004061,1.043483
3,1.268941,0.756176,0.995859,0.698525,1.219313,1.312310,-0.105884,1.019816,0.125426,1.100298,...,-0.258336,1.275332,0.222033,1.530298,0.013343,1.331226,1.263994,1.566393,1.007431,0.936340
4,1.244109,-0.471216,0.998289,0.843386,0.934412,0.919458,-0.060159,0.987376,0.141403,1.365036,...,-0.244661,1.209525,-0.069025,1.617372,0.006191,1.369306,0.111701,-0.275220,1.006267,1.046875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,-0.475462,0.432039,-0.501959,1.526507,0.655333,-0.769226,-1.283057,-0.851739,1.020212,-0.418085,...,0.910433,-0.084863,0.237903,1.268608,-0.990685,0.239095,0.380915,0.807177,-0.884386,0.452602
9996,-0.176707,1.194638,-0.666174,-0.293345,0.628074,-1.630229,-0.960342,-0.808003,0.796632,-0.119920,...,1.299141,-0.242049,-0.217186,0.353995,-1.018035,0.169191,0.283019,0.689421,-0.912231,0.483123
9997,-0.653051,1.249559,-0.468456,1.590317,0.554801,-1.236882,-1.151610,-0.850651,0.324139,-0.289302,...,-0.050079,0.050049,-0.757661,1.251994,-0.987379,-0.241169,-0.539540,0.965077,-0.921871,0.263013
9998,-0.416609,2.005280,-0.782373,0.227326,0.528569,-1.219151,-0.841334,-0.788057,0.975515,-0.082027,...,0.228491,-0.311175,-0.040485,0.714919,-0.995424,-0.067425,-1.882409,0.446363,-0.907938,0.521534


In [13]:
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.str.replace('.', '').to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

alt.Chart(...)

In [14]:
signals_len = ds[0][1].shape[-1]
signals_len

10000

In [15]:
parameters['model'] = {
    'input_size': signals_len,
    'n_classes': len(ds.labels_names),
    'kernel_sizes': [128, 64, 32],
    'channels': [1, 12, 12, 12],
    'stride': 2,
    'dropout_rate': 0.25,
}

In [16]:
x = batch[1][0].numpy()
model = trapallada.model.signal.Classifier(**parameters['model'], key=jr.PRNGKey(42))
model.debug_call(x, jr.PRNGKey(42))
print(model)

conv_output_len=(4937,)
conv_output_len=(2437,)
conv_output_len=(1203,)
14436
after equinox.nn._conv: (12, 4937)
after jax._src.nn.functions: (12, 4937)
after equinox.nn._normalisation: (12, 4937)
after equinox.nn._dropout: (12, 4937)
after equinox.nn._conv: (12, 2437)
after jax._src.nn.functions: (12, 2437)
after equinox.nn._normalisation: (12, 2437)
after equinox.nn._dropout: (12, 2437)
after equinox.nn._conv: (12, 1203)
after jax._src.nn.functions: (12, 1203)
after equinox.nn._normalisation: (12, 1203)
after equinox.nn._dropout: (12, 1203)
after jax._src.numpy.lax_numpy: (14436,)
after equinox.nn._linear: (4,)
Classifier(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,1,128],
      bias=f32[12,1],
      in_channels=1,
      out_channels=12,
      kernel_size=(128,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(12, 4937),
      eps=1e-05,
     

In [17]:
def loss(model: eqx.Module, x: Float[Array, 'batch length'], y: Float[Array, 'batch'], key: PRNGKeyArray):

    y_pred = jax.vmap(model)(x, jr.split(key, len(x)))
    
    return optax.losses.softmax_cross_entropy_with_integer_labels(y_pred, y).mean()

In [18]:
@eqx.filter_jit
def take_step(
    model,
    opt_state: PyTree,
    x: Float[Array, 'batch 1 length'],
    y: Float[Array, 'batch'],
    key: PRNGKeyArray
) -> tuple[tuple[eqx.Module], PyTree, Float[Array, '']]:

    value, grads = eqx.filter_value_and_grad(loss)(model, x, y, key)

    # it returns transformed/processed gradients, and the optimizer state
    updates, opt_state = optim.update(grads, opt_state, model)

    model = eqx.apply_updates(model, updates)

    return model, opt_state, value

In [19]:
def make_results_df(n_epochs: int) -> pd.DataFrame:
    
    return pd.DataFrame({ (data, metric): [0.] * n_epochs for data in ['training', 'validation'] for metric in ['loss', 'accuracy']})

make_results_df(2)

training          validation         
      loss accuracy       loss accuracy
0      0.0      0.0        0.0      0.0
1      0.0      0.0        0.0      0.0

In [20]:
@eqx.filter_jit
def compute_accuracy(model: eqx.Module, x: Float[Array, "batch 1 500"], y: Float[Array, "batch"], key: PRNGKeyArray) -> Float[Array, ""]:
    
    y_pred = jnp.argmax(jax.nn.log_softmax(jax.vmap(model)(x, jr.split(key, len(x)))), axis=-1)
    
    return (y_pred == y).mean()

In [21]:
def evaluate(model: eqx.Module, dataloader: torch.utils.data.DataLoader):

    # a fixed key is used since we are not training
    key = jr.PRNGKey(42)

    # in case, there are layers that behave differently in training and inference modes
    model = eqx.nn.inference_mode(model)

    loss_acc = 0.
    accuracy_acc = 0.

    for _, x, y in dataloader:

        x = x.numpy()
        y = y.numpy()
    
        loss_value = loss(model, x, y, key)
        
        # adding up *averages* (of both the loss and the accuracy)
        loss_acc += loss_value.item()
        accuracy_acc += compute_accuracy(model, x, y, key).item()

    # reverting to training mode
    model = eqx.nn.inference_mode(model, value=False)

    return loss_acc / len(dataloader), accuracy_acc / len(dataloader)


In [26]:
parameters['model'] = {
    'input_size': signals_len,
    'n_classes': len(ds.labels_names),
    'kernel_sizes': [128, 64, 32],
    'channels': [1, 4, 4, 4],
    # 'channels': [1, 6, 6, 6],
    # 'channels': [1, 8, 8, 8], # <- best for the initial dataset
    # 'channels': [1, 10, 10, 10],
    # 'channels': [1, 12, 12, 12],
    # 'stride': 2,
    'stride': 1,
    'dropout_rate': 0.,
    # 'dropout_rate': 0.25, # <- best for the initial dataset
}

In [27]:
model = trapallada.model.signal.Classifier(**parameters['model'], key=jr.PRNGKey(42))
optim = optax.adamw(parameters['training']['learning_rate'])
opt_state = optim.init(eqx.filter(model, eqx.is_array))

conv_output_len=(9873,)
conv_output_len=(9810,)
conv_output_len=(9779,)
39116


In [28]:
# parameters['training']['n_epochs'] = 100
parameters['training']['n_epochs'] = 20

results_df = make_results_df(parameters['training']['n_epochs'])

for epoch in range(parameters['training']['n_epochs']):

    for _, x, y in train_dl:

        key, key2, key3 = jr.split(key, 3)

        x = x.numpy()
        y = y.numpy()

        model, opt_state, loss_value = take_step(model, opt_state, x, y, key2)

        # adding up *averages* (of both the loss and the accuracy)
        results_df.loc[epoch, ('training', 'loss')] += loss_value.item()
        results_df.loc[epoch, ('training', 'accuracy')] += compute_accuracy(model, x, y, key3)


    results_df.loc[epoch, [('training', 'loss'), ('training', 'accuracy')]] /= len(train_dl)

    # validation
    results_df.loc[epoch, [('validation', 'loss'), ('validation', 'accuracy')]] = evaluate(model, validation_dl)

    print(
        epoch,
        results_df.loc[epoch, ('training', 'loss')],
        results_df.loc[epoch, ('validation', 'loss')],
        '|||',
        results_df.loc[epoch, ('training', 'accuracy')],
        results_df.loc[epoch, ('validation', 'accuracy')],
    )

0 0.9967627139425335 0.3524963898035799 ||| 0.9117307692307692 0.942454268292683
1 0.36621255164157274 0.26583369327431006 ||| 0.9803846153846154 0.9676067073170732
2 0.10949295666030745 0.6324454268586214 ||| 0.9965384615384615 0.9416920731707317
3 0.22566753710801404 0.13058448619816673 ||| 0.9914423076923077 0.9855182926829268
4 0.2568275499535041 1.923350767145931 ||| 0.9924038461538461 0.9089176829268293
5 0.2849298607514914 0.12342166220984786 ||| 0.9925961538461539 0.9866615853658537
6 0.4560213332511998 0.14624633165028542 ||| 0.9874038461538461 0.9885670731707317
7 0.07012788806510498 0.10307899665747894 ||| 0.9984615384615385 0.9939024390243902
8 0.03463405580934125 0.20222528096270478 ||| 0.9996153846153846 0.9870426829268293
9 0.13018807746861816 0.04861011647305351 ||| 0.9975 0.995045731707317
10 0.3411933569784421 0.20341634014199364 ||| 0.9935576923076923 0.9891768294136699
11 0.16205942516657695 0.2402703087828169 ||| 0.9976923076923077 0.988033536730743
12 0.0850801470

In [25]:
loss_df = results_df.loc[:epoch-1,[('training', 'loss'), ('validation', 'loss')]]
# loss_df.columns = ["_".join(a) for a in loss_df.columns.to_flat_index()]
loss_df.columns = loss_df.columns.get_level_values(0)
loss_chart = alt.Chart(
    loss_df.reset_index()
).transform_fold(
    loss_df.columns.to_list(), ['dataset', 'loss']
).mark_line().encode(
    x='index:Q', y='loss:Q', color='dataset:N'
).properties(width=600, height=400)
# ------
accuracy_df = results_df.loc[:epoch-1,[('training', 'accuracy'), ('validation', 'accuracy')]]
# accuracy_df.columns = ["_".join(a) for a in accuracy_df.columns.to_flat_index()]
accuracy_df.columns = accuracy_df.columns.get_level_values(0)
accuracy_df_chart = alt.Chart(
    accuracy_df.reset_index()
).transform_fold(
    accuracy_df.columns.to_list(), ['dataset', 'loss']
).mark_line().encode(
    x='index:Q', y='loss:Q', color='dataset:N'
).properties(width=600, height=400)
# ------
# loss_chart
# accuracy_df_chart
loss_chart | accuracy_df_chart

alt.HConcatChart(...)